In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment yolo-train with packages, registered as a Jupyter kernel.
''''
conda create -n yolo-train python=3.10 -y
conda activate yolo-train

conda install -y -c conda-forge \
  ipykernel numpy pandas pyyaml tqdm matplotlib scikit-learn psutil

pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
  torch torchvision torchaudio

pip install --no-cache-dir \
  ultralytics opencv-python-headless seaborn tensorboard ipywidgets "faster-coco-eval>=1.6.7"

pip install statsmodels

python -m ipykernel install --user --name yolo-train --display-name "yolo-train"
'''

In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import os
import sys
import time
import math
import random
import platform
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import ultralytics
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [ ]:
# === Cell 4: Config ===

# --- Model weight paths (add or remove entries as needed) ---
MODEL_PATHS = [
    Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260626_Yolo_Minimal_Training/20260626_Yolo26x_Minimal_Training/runs/train/weights/mt_yolo26x-seg_best.pt"),
    Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260626_Yolo_Minimal_Training/20260626_Yolov8x_Minimal_Training/runs/train/weights/mt_yolov8x-seg_best.pt"),
    Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260626_Yolo_Minimal_Training/20260626_Yolov8n_Minimal_Training/runs/train/weights/mt_yolov8n-seg_best.pt"),
    Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260603_Yolo_Deployment_Training/runs/20260603_Yolo_Deployment_Training_DeploymentSegTraining/weights/best.pt"),

]

# --- Image folder for inference ---
#IMAGES_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/spectralwaste_rgb/dataset_flattened")
IMAGES_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Initial_Image_Dataset/1080_resolution")

# --- Ground-truth labels (YOLO segmentation .txt files) ---
LABELS_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/labels/all")

# --- Output directory for saved visualizations ---
OUTPUT_DIR = Path("./20260603_Yolo_Inference_Visualization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Dataset semantics ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}

# --- Inference defaults ---
CONF_THRESHOLD = 0.50
IOU_THRESHOLD  = 0.50
IMAGE_SIZE     = 640
MAX_DET        = 300

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
# === Cell 5: Environment and Path Checks ===

print("Environment")
print("-" * 80)
print(f"Python:              {sys.version}")
print(f"Platform:            {platform.platform()}")
print(f"torch version:       {torch.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count:   {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"CUDA device {i}:       {torch.cuda.get_device_name(i)}")
    DEVICE = 0
else:
    print("No GPU detected — using CPU for inference.")
    DEVICE = "cpu"
print(f"ultralytics version: {ultralytics.__version__}")
print(f"Selected device:     {DEVICE}")

print()
print("Path checks")
print("-" * 80)

def _check(label, path, kind="dir"):
    tag = "DIR " if kind == "dir" else "FILE"
    ok  = path.is_dir() if kind == "dir" else path.is_file()
    status = "OK" if ok else "MISSING"
    print(f"  [{tag}] {status:8s} {label}: {path}")
    return ok

all_ok = True
all_ok &= _check("IMAGES_DIR", IMAGES_DIR, "dir")
all_ok &= _check("LABELS_DIR", LABELS_DIR, "dir")
all_ok &= _check("OUTPUT_DIR", OUTPUT_DIR, "dir")

print()
print("Model weight files")
print("-" * 80)
valid_models = []
for i, mp in enumerate(MODEL_PATHS):
    exists = mp.is_file()
    status = "OK" if exists else "MISSING"
    print(f"  [{i}] {status:8s} {mp}")
    if exists:
        valid_models.append(mp)
    all_ok &= exists

if not valid_models:
    print("\nERROR: No valid model weights found. Check MODEL_PATHS in the config cell.")
elif not all_ok:
    print("\nWARNING: Some paths are missing — only valid models will be available.")
else:
    print("\nAll paths verified.")

In [ ]:
# === Cell 6: Image Pool and Model Loading ===

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

image_pool = sorted([
    p for p in IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

print(f"Found {len(image_pool)} images in {IMAGES_DIR}")
if len(image_pool) == 0:
    raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

# --- Load all valid models once ---
loaded_models = {}
model_labels  = []

for mp in valid_models:
    # Display label: the three parent folders plus the weight filename
    parts  = mp.parts
    label  = "/".join(parts[-4:-1]) if len(parts) >= 4 else mp.stem
    label  = f"{label}  ({mp.name})"

    print(f"Loading: {label}")
    model = YOLO(str(mp))
    loaded_models[label] = {"model": model, "path": mp}
    model_labels.append(label)

print(f"\nLoaded {len(loaded_models)} model(s), {len(image_pool)} image(s) ready.")

In [ ]:
# === Cell 7: Interactive Inference Viewer ===

# --- Color palettes (shared with the training and screening notebooks) ---
CLASS_COLORS_RGB = {          # prediction colors  (solid, vivid)
    0: (255,   0,   0),       # body  — red
    1: (  0, 200,   0),       # cap   — green
    2: (  0,  80, 255),       # label — blue
}
GT_COLORS_RGB = {             # ground-truth colors (lighter pastels)
    0: (255, 160, 160),       # body  — light pink/red
    1: (160, 255, 160),       # cap   — light green
    2: (160, 200, 255),       # label — light blue
}
DEFAULT_COLOR_RGB  = (255, 255,   0)
DEFAULT_GT_RGB     = (255, 255, 160)
MASK_ALPHA         = 0.35
BOX_THICKNESS      = 2
TEXT_SCALE         = 0.55
TEXT_THICKNESS     = 2

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

def _rgb2bgr(c):  return (c[2], c[1], c[0])
def _cls_bgr(cid):   return _rgb2bgr(CLASS_COLORS_RGB.get(int(cid), DEFAULT_COLOR_RGB))
def _gt_bgr(cid):    return _rgb2bgr(GT_COLORS_RGB.get(int(cid), DEFAULT_GT_RGB))

# --- YOLO label parser ---
def parse_yolo_label_line(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id = int(float(parts[0]))
    coords = [float(v) for v in parts[1:]]
    polygon = list(zip(coords[0::2], coords[1::2]))
    return {"class_id": cls_id, "polygon_norm": polygon}

# --- Dashed polygon helper (OpenCV has no dashed line) ---
def draw_dashed_polygon(img, points, color, thickness=2, dash_px=12, gap_px=7):
    """Draw a closed dashed polygon on img in-place."""
    pts = list(points)
    pts.append(pts[0])                          # close the loop
    for i in range(len(pts) - 1):
        x0, y0 = int(pts[i][0]),   int(pts[i][1])
        x1, y1 = int(pts[i+1][0]), int(pts[i+1][1])
        dx, dy  = x1 - x0, y1 - y0
        seg_len = max(1, int(np.hypot(dx, dy)))
        step    = dash_px + gap_px
        t       = 0
        while t < seg_len:
            t_end   = min(t + dash_px, seg_len)
            sx = int(round(x0 + dx * t     / seg_len))
            sy = int(round(y0 + dy * t     / seg_len))
            ex = int(round(x0 + dx * t_end / seg_len))
            ey = int(round(y0 + dy * t_end / seg_len))
            cv2.line(img, (sx, sy), (ex, ey), color, thickness, cv2.LINE_AA)
            t += step

# --- Renderer (predictions solid, GT dashed) ---
def draw_overlay(image_path, result, show_masks=True, show_contours=True,
                 show_boxes=False, show_labels=True, show_gt=True):
    """
    Render predictions (solid) and GT polygons from LABELS_DIR (dashed) on one image.
    """
    img_bgr = result.orig_img.copy()
    overlay  = img_bgr.copy()
    boxes    = result.boxes
    masks    = result.masks
    h, w     = img_bgr.shape[:2]

    # --- Filled prediction masks ---
    if show_masks and masks is not None and masks.xy is not None:
        for i, poly in enumerate(masks.xy):
            if poly is None or len(poly) < 3: continue
            cls_id = int(boxes.cls[i].item()) if boxes is not None and i < len(boxes) else -1
            pts    = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _cls_bgr(cls_id))
    blended = cv2.addWeighted(overlay, MASK_ALPHA, img_bgr, 1 - MASK_ALPHA, 0)

    # --- Prediction contours, boxes and labels ---
    if boxes is not None and len(boxes):
        xyxy    = boxes.xyxy.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        polys   = masks.xy if masks is not None and masks.xy is not None else []

        for i in range(len(xyxy)):
            cls_id     = int(cls_ids[i])
            conf       = float(confs[i])
            color      = _cls_bgr(cls_id)
            x1,y1,x2,y2 = xyxy[i].astype(int)

            if show_contours and i < len(polys):
                poly = polys[i]
                if poly is not None and len(poly) >= 3:
                    pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
                    cv2.polylines(blended, [pts], True, color, 2, cv2.LINE_AA)

            if show_boxes:
                cv2.rectangle(blended, (x1,y1), (x2,y2), color, BOX_THICKNESS)

            if show_labels:
                name  = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
                label = f"{name} {conf:.2f}"
                (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                               TEXT_SCALE, TEXT_THICKNESS)
                ly1 = max(0, y1 - th - bl - 4)
                cv2.rectangle(blended, (x1, ly1), (x1 + tw + 6, y1), color, -1)
                cv2.putText(blended, label, (x1 + 3, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                            (255,255,255), TEXT_THICKNESS, cv2.LINE_AA)

    # --- Dashed GT polygon outlines ---
    if show_gt:
        label_path = LABELS_DIR / (Path(image_path).stem + ".txt")
        if label_path.exists():
            with label_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parsed = parse_yolo_label_line(line)
                    if parsed is None: continue
                    pts_px = [
                        (int(round(x * w)), int(round(y * h)))
                        for x, y in parsed["polygon_norm"]
                    ]
                    if len(pts_px) < 3: continue
                    gt_color = _gt_bgr(parsed["class_id"])
                    draw_dashed_polygon(blended, pts_px, gt_color,
                                        thickness=2, dash_px=12, gap_px=7)

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

# --- Widget controls ---
if not loaded_models:
    print("No models loaded. Check MODEL_PATHS in the config cell.")
elif len(image_pool) == 0:
    print("No images found. Check IMAGES_DIR in the config cell.")
else:
    # Model selector
    model_dropdown = widgets.Dropdown(
        options=[(lbl, lbl) for lbl in model_labels],
        value=model_labels[0],
        description="Model:",
        layout=widgets.Layout(width="700px"),
    )

    # Image dropdown
    def _make_img_opts(pool):
        return [(f"{i}: {p.name}", i) for i, p in enumerate(pool)]

    image_dropdown = widgets.Dropdown(
        options=_make_img_opts(image_pool),
        description="Image:",
        layout=widgets.Layout(width="700px"),
    )

    # Buttons
    show_button   = widgets.Button(description="Show",    button_style="primary", icon="eye")
    random_button = widgets.Button(description="Random",  button_style="info",    icon="random")
    save_button   = widgets.Button(description="Save PNG",                         icon="save")

    # Sliders
    conf_slider = widgets.FloatSlider(
        value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.01,
        description="Conf:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )
    iou_slider = widgets.FloatSlider(
        value=IOU_THRESHOLD, min=0.10, max=0.90, step=0.05,
        description="IoU:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )

    # Checkboxes
    show_masks_cb    = widgets.Checkbox(value=True,  description="Masks")
    show_contours_cb = widgets.Checkbox(value=True,  description="Contours")
    show_boxes_cb    = widgets.Checkbox(value=False, description="Boxes")
    show_labels_cb   = widgets.Checkbox(value=True,  description="Labels")
    show_gt_cb       = widgets.Checkbox(value=True,  description="Ground truth (dashed)")

    viewer_output = widgets.Output()

    # --- Render callback ---
    def _render(save_path=None):
        idx = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(image_pool):
            return

        image_path  = image_pool[idx]
        model_label = model_dropdown.value
        model       = loaded_models[model_label]["model"]

        # --- Timed inference ---
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t_start = time.perf_counter()

        results = model.predict(
            source=str(image_path),
            conf=conf_slider.value,
            iou=iou_slider.value,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            retina_masks=True,
            verbose=False,
            save=False,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t_end = time.perf_counter()
        inference_ms = (t_end - t_start) * 1000.0

        r      = results[0]
        n_pred = 0 if r.boxes is None else len(r.boxes)

        rgb = draw_overlay(
            image_path, r,
            show_masks=show_masks_cb.value,
            show_contours=show_contours_cb.value,
            show_boxes=show_boxes_cb.value,
            show_labels=show_labels_cb.value,
            show_gt=show_gt_cb.value,
        )

        device_tag = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"

        with viewer_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.imshow(rgb)
            ax.set_title(
                f"{image_path.name}  |  pred={n_pred}  |  "
                f"conf\u2265{conf_slider.value:.2f}  |  "
                f"inference={inference_ms:.1f} ms ({device_tag})\n"
                f"solid=prediction  dashed=GT  |  "
                f"body=red  cap=green  label=blue  (GT = lighter shade)",
                fontsize=9,
            )
            ax.axis("off")
            plt.tight_layout()
            if save_path:
                fig.savefig(save_path, bbox_inches="tight", dpi=150)
                print(f"Saved: {save_path}")
            plt.show()

    # --- Button callbacks ---
    def on_show_clicked(b):
        _render()

    def on_random_clicked(b):
        image_dropdown.value = random.randrange(len(image_pool))
        _render()

    def on_save_clicked(b):
        idx = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(image_pool):
            return
        img_stem    = image_pool[idx].stem
        model_tag   = Path(loaded_models[model_dropdown.value]["path"]).parts[-3]
        save_path   = OUTPUT_DIR / f"infer_{model_tag}_{img_stem}.png"
        _render(save_path=save_path)

    show_button.on_click(on_show_clicked)
    random_button.on_click(on_random_clicked)
    save_button.on_click(on_save_clicked)

    # --- Layout ---
    display(
        widgets.VBox([
            widgets.HBox([model_dropdown]),
            widgets.HBox([image_dropdown]),
            widgets.HBox([show_button, random_button, save_button]),
            widgets.HBox([conf_slider, iou_slider]),
            widgets.HBox([show_masks_cb, show_contours_cb, show_boxes_cb,
                          show_labels_cb, show_gt_cb]),
        ]),
        viewer_output,
    )